# LELA60331 Week 10 Seminar: An intent classifier

In Week 8 we trained a softmax classifier on simulated data. This week we apply it to a real language task, **intent classification**, and look at three things that matter when training on real data: keeping a separate **development set**, **monitoring** performance during training, and training in **mini-batches**.

| Part | Topic |
|---|---|
| 1 | The intent classification data |
| 2 | Training and evaluating a softmax classifier |
| 3 | Monitoring training with the development set |
| 4 | Mini-batch training |

---
# Part 1 · The intent classification data

A spoken dialogue system (a voice assistant, for example) first needs to work out what the speaker wants to do: their **intent**. Only then can it produce a suitable response. Our data set pairs 2100 utterances with one of seven intents:

| intent | example |
|---|---|
| PlayMusic | "play easy listening" |
| AddToPlaylist | "please add this song to road trip" |
| RateBook | "give this novel 5 stars" |
| SearchScreeningEvent | "give me a list of local movie times" |
| BookRestaurant | "i'd like a table for four at 7pm at Asti" |
| GetWeather | "what's it like outside" |
| SearchCreativeWork | "show me the new James Bond trailer" |

Run the next two cells to download the data and prepare it. The preparation works just like the Yelp data in Week 7:

* each utterance becomes a one-hot row over the 2500 most frequent words, plus an extra column of 1s for the bias trick (Week 8)
* each intent becomes a one-hot vector over the seven intents

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/intent_classification.csv

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=3)

utterances = []
labels = []
with open("intent_classification.csv", encoding="utf-8-sig") as fp:
    lines = fp.readlines()
for line in lines[1:]:                          # skip the header line
    fields = line.rstrip().split(",")           # [utterance, intent]
    utterances.append(fields[0])
    labels.append(fields[1])

# the vocabulary: the 2500 most frequent words
tokenized = [u.split() for u in utterances]
counts = {}
for sent in tokenized:
    for word in sent:
        counts[word] = counts.get(word, 0) + 1
pairs = sorted([(count, word) for word, count in counts.items()], reverse=True)
type_list = [word for count, word in pairs[:2500]]
word_to_index = {}
for j in range(len(type_list)):
    word_to_index[type_list[j]] = j

def one_hot_utterance(words):
    """A one-hot row for a list of words, plus the extra 1 at the end."""
    row = np.zeros(len(type_list) + 1)
    row[-1] = 1
    for word in words:
        if word in word_to_index:
            row[word_to_index[word]] = 1
    return row

X = np.array([one_hot_utterance(words) for words in tokenized])

# the intents, and their one-hot vectors
intents = sorted(set(labels))
intent_to_index = {}
for k in range(len(intents)):
    intent_to_index[intents[k]] = k
Y = np.eye(len(intents))[[intent_to_index[label] for label in labels]]

print(X.shape, Y.shape)
print(intents)

### Training, development and test sets

So far we have split our data into a **training** set and a **test** set. In real projects we usually make a third set, the **development** set (or *validation* set). We use it while we are building the model: to watch how training is going, and to compare different settings, such as learning rates. The test set is kept aside until the very end, so that it gives an honest estimate of how well the final model does on data it has never seen. If we kept checking the test set while choosing settings, we would gradually tune the model to the test set itself.

We shuffle the positions 0 to 2099 and use 80% for training (1680 utterances), 10% for development (210) and 10% for testing (210).

### 🏋️ Exercise 1 — Splitting the data

Complete the split. `order[:1680]` gives the first 1680 shuffled positions. Then use the position arrays to select rows: `X[train_ints]` gives the rows of `X` for the training utterances.

In [ ]:
np.random.seed(1)
order = np.random.permutation(len(utterances))
train_ints = order[:1680]
dev_ints = ???
test_ints = ???

X_train, Y_train = X[train_ints], Y[train_ints]
X_dev, Y_dev = ???
X_test, Y_test = ???
print(X_train.shape, X_dev.shape, X_test.shape)     # (1680, 2501) (210, 2501) (210, 2501)

---
# Part 2 · Training and evaluating a softmax classifier

### 🏋️ Exercise 2 — Training

Complete the training loop. It is the same as in Week 8, Exercise 3, but with the intent data: `X_train` and `Y_train` instead of `X` and `y`. Training takes about half a minute.

In [ ]:
def softmax_rows(z):
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

np.random.seed(10)
num_samples, num_features = X_train.shape
num_classes = len(intents)
weights = np.random.rand(num_features, num_classes)
n_iters = 300
lr = 1.0
ce_loss = []

for i in range(n_iters):
    z = ???
    q = ???
    loss = np.mean(-np.log2(np.sum(Y_train * q, axis=1)))
    ce_loss.append(loss)
    dw = ???
    weights = ???

plt.plot(range(1, n_iters), ce_loss[1:])
plt.xlabel("number of epochs")
plt.ylabel("loss")
plt.show()

### 🏋️ Exercise 3 — Evaluation

Here is the function for macro-averaged precision and recall from Week 8.

1. Write `predict(X_part)`, which returns the predicted intent number for every row of `X_part`.
2. Use it to calculate the accuracy and the macro-averaged precision and recall on the **test** set. *Hint:* the true intent numbers are `np.argmax(Y_test, axis=1)`.

In [ ]:
def macro_precision_recall(y_pred, y_true, num_classes):
    precisions = []
    recalls = []
    for k in range(num_classes):
        tp = np.sum((y_pred == k) & (y_true == k))
        fp = np.sum((y_pred == k) & (y_true != k))
        fn = np.sum((y_pred != k) & (y_true == k))
        precisions.append(tp / (tp + fp))
        recalls.append(tp / (tp + fn))
    return float(np.mean(precisions)), float(np.mean(recalls))

def predict(X_part):
    return ???

y_test_pred = ???
y_test_true = ???
print("test accuracy:", ???)
print("test macro precision and recall:", ???)

### 🏋️ Exercise 4 — Try it out

Write `predict_intent(utterance)`, which returns the predicted intent **name** for a new utterance. Use `one_hot_utterance` (from the preparation cell) on the utterance's words, then the trained weights. Try some utterances of your own. Which kinds of utterance confuse it?

In [ ]:
def predict_intent(utterance):
    row = ???
    probs = ???
    return ???

for u in ["play some jazz", "will it rain in manchester tomorrow", "book a table for two tonight",
          "add this track to my running playlist", "i would rate this book four stars"]:
    print(u, "->", predict_intent(u))

---
# Part 3 · Monitoring training with the development set

The loss curve shows how well the model fits the **training** data. But what we really care about is how well it does on data it has **not** been trained on. A model can keep getting better on the training data while getting no better, or even worse, on new data: it starts to learn quirks of the training data that don't generalise. This is **overfitting** (Week 5). The development set lets us watch for it.

### 🏋️ Exercise 5 — Accuracy during training

Retrain the model from scratch. Every 10 epochs, calculate the accuracy on the training set and on the development set (using `predict`), and append them to `train_acc` and `dev_acc`. The plot then shows both curves.

Do the two curves level off at the same point? Is there any sign of overfitting? How many epochs would you choose?

In [ ]:
np.random.seed(10)
weights = np.random.rand(num_features, num_classes)
train_acc = []
dev_acc = []
checked_at = []

for i in range(n_iters):
    q = softmax_rows(X_train.dot(weights))
    weights = weights - lr * X_train.T.dot(q - Y_train) / num_samples
    if i % 10 == 0:                     # every 10 epochs
        checked_at.append(i)
        train_acc.append(???)
        dev_acc.append(???)

plt.plot(checked_at, train_acc, label="training")
plt.plot(checked_at, dev_acc, label="development")
plt.xlabel("number of epochs")
plt.ylabel("accuracy")
plt.legend()
plt.show()

---
# Part 4 · Mini-batch training

So far, every update of the weights has used **all** the training data: one update per pass through the data (one **epoch**). With large data sets this is slow, because we look at every item before making a single small step.

In **mini-batch training** we shuffle the training data, split it into small **batches** (say 32 items each), and update the weights after **each batch**. One epoch then makes many small updates instead of one. Each update is based on less data, so it is a little noisier, but the model usually learns much faster. This is how almost all neural networks are trained.

### 🏋️ Exercise 6 — Mini-batch training

Complete the inner loop. For each batch:

1. select the batch's rows: `X_train[batch]` and `Y_train[batch]`
2. calculate the probabilities and the gradient for the batch, dividing by the number of items **in the batch**
3. update the weights

How does the accuracy after 20 epochs compare with full-batch training after 300? And the time taken?

In [ ]:
import time
np.random.seed(10)
weights = np.random.rand(num_features, num_classes)
batch_size = 32
n_epochs = 20
lr = 0.5
batch_loss = []
start_time = time.time()

for epoch in range(n_epochs):
    order = np.random.permutation(num_samples)          # shuffle each epoch
    epoch_loss = 0
    for start in range(0, num_samples, batch_size):
        batch = order[start:start + batch_size]          # the positions in this batch
        X_batch = ???
        Y_batch = ???
        q = ???
        epoch_loss += np.sum(-np.log2(np.sum(Y_batch * q, axis=1)))
        dw = ???
        weights = ???
    batch_loss.append(epoch_loss / num_samples)

print("time:", round(time.time() - start_time, 1), "seconds")
plt.plot(range(n_epochs), batch_loss)
plt.xlabel("number of epochs")
plt.ylabel("loss")
plt.show()

y_test_pred = predict(X_test)
print("test accuracy:", np.mean(y_test_pred == y_test_true))
print("test macro precision and recall:", macro_precision_recall(y_test_pred, y_test_true, num_classes))

⭐ **Challenge (optional):** try batch sizes of 1, 8, 128 and 1680 (the whole training set). How do the time, the shape of the loss curve and the final accuracy change?

⭐ **Challenge (optional):** for each intent, print its precision and recall on the test set. Which intents are hardest to tell apart? Look at some misclassified test utterances (`utterances[test_ints[i]]`) to see why.

---
## ✏️ Summary

This week you have:

* trained and evaluated a softmax classifier for a real seven-class task
* used a **development set** to monitor training, keeping the **test set** for the final evaluation
* trained with **mini-batches**, updating the weights after each small batch of data